In [1]:
import sys
print(sys.executable)

c:\Users\ramti\anaconda3\envs\osv5m\python.exe


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
CODES_DIR = PROJECT_ROOT / "codes"
OSV5M_REPO = CODES_DIR / "github-osv5m" / "osv5m"
BASELINE_PATH = CODES_DIR / "baseline-huggingface" / "baseline"
SUBSET_DIR = PROJECT_ROOT / "datasets" / "subset_test"
sys.path.append(str(OSV5M_REPO))

from PIL import Image
from models.huggingface import Geolocalizer


In [ ]:
import os
os.chdir(OSV5M_REPO)
print("cwd الان:", os.getcwd())


cwd الان: C:\Users\ramti\Desktop\ramtin legion pc\projects\myproject\codes\github-osv5m\osv5m


In [ ]:
# Optional one-time conversion of the downloaded model weights.
# Use BASELINE_PATH from the setup cells instead of a machine-specific path.
# import torch
# from safetensors.torch import save_file
# bin_path = BASELINE_PATH / "pytorch_model.bin"
# safetensors_path = BASELINE_PATH / "model.safetensors"
# state_dict = torch.load(bin_path, map_location="cpu", weights_only=True)
# state_dict = {k: v.clone().contiguous() for k, v in state_dict.items()}
# save_file(state_dict, safetensors_path)


تبدیل با موفقیت انجام شد ✅


In [ ]:
geoloc = Geolocalizer.from_pretrained(BASELINE_PATH)
img = Image.open(".media/examples/img1.jpeg")
x = geoloc.transform(img).unsqueeze(0)
gps = geoloc(x)
print(gps)  # (lat, lon) به رادیان


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] CLIPVisionModel LOAD REPORT from: laion/CLIP-ViT-L-14-DataComp.XL-s13B-b90K
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
visual_projection.weight                                     | UNEXPECTED |  | 
text_model.encoder.layers.{0.

Loading weights from local directory
tensor([[0.5692, 2.3100]], dtype=torch.float64, grad_fn=<MulBackward0>)


In [ ]:
import os
from pathlib import Path

DATA_ROOT = Path(os.environ.get("OSV5M_DATA_DIR", PROJECT_ROOT / "data" / "osv5m"))
test_img_dir = DATA_ROOT / "images" / "test" / "extracted" / "01"
files = os.listdir(test_img_dir)
print("تعداد کل:", len(files))
print("۵ نمونه:", files[:5])


تعداد کل: 50000
۵ نمونه: ['1000018734153694.jpg', '1000206250513605.jpg', '1000213977388555.jpg', '1000316360541560.jpg', '1000509413686600.jpg']


In [ ]:
# The old subset-generation experiment is now maintained in codes/prepare_subset.py.
# It uses the shared project paths and OSV5M_DATA_DIR configuration.


تعداد کل رکورد در test.csv: 210122
تعداد عکس واقعی پیدا شده (کل zipها): 210122
تعداد رکورد متادیتا که عکسشون موجوده: 210122
تعداد نهایی subset: 600
تعداد کشورهای متفاوت: 207
✅ subset ساخته شد در: C:\Users\ramti\OneDrive\Desktop\ramtin legion pc\projects\myproject\datasets\subset_test


In [ ]:
# prepare_subset.py (نسخه‌ی جدید 1095 تایی، اندازه‌ی subset قابل تنظیم)

import pandas as pd
import os
import shutil
import math

DATA_ROOT = Path(os.environ.get("OSV5M_DATA_DIR", PROJECT_ROOT / "data" / "osv5m"))
TEST_CSV = DATA_ROOT / "test.csv"
TEST_IMG_DIR = DATA_ROOT / "images" / "test" / "extracted"
OUTPUT_DIR = PROJECT_ROOT / "datasets" / "subset_test"

MAX_TOTAL = 1100
RANDOM_SEED = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR / "images", exist_ok=True)

df = pd.read_csv(TEST_CSV, low_memory=False)
print("تعداد کل رکورد در test.csv:", len(df))

available_files = {}
for root, dirs, files in os.walk(TEST_IMG_DIR):
    for f in files:
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            file_id = os.path.splitext(f)[0]
            available_files[file_id] = os.path.join(root, f)

print("تعداد عکس واقعی پیدا شده:", len(available_files))

df["id_str"] = df["id"].astype(str)
df_avail = df[df["id_str"].isin(available_files.keys())].copy()
print("تعداد رکورد متادیتا که عکسشون موجوده:", len(df_avail))

N_PER_COUNTRY = math.ceil(MAX_TOTAL / df_avail["country"].nunique()) + 1
subset_list = []
for country, grp in df_avail.groupby("country"):
    n = min(len(grp), N_PER_COUNTRY)
    subset_list.append(grp.sample(n, random_state=RANDOM_SEED))
df_avail = pd.concat(subset_list, ignore_index=True)

if len(df_avail) > MAX_TOTAL:
    df_avail = df_avail.sample(MAX_TOTAL, random_state=RANDOM_SEED)

print("تعداد نهایی subset:", len(df_avail))
print("تعداد کشورهای متفاوت:", df_avail["country"].nunique())

for _, row in df_avail.iterrows():
    src = available_files[row["id_str"]]
    dst = OUTPUT_DIR / "images" / os.path.basename(src)
    shutil.copy2(src, dst)

cols_to_keep = ["id", "latitude", "longitude", "country", "region", "sub-region", "city"]
df_avail[cols_to_keep].to_csv(OUTPUT_DIR / "subset_metadata.csv", index=False)

print("✅ subset ساخته شد در:", OUTPUT_DIR)


تعداد کل رکورد در test.csv: 210122
تعداد عکس واقعی پیدا شده: 210122
تعداد رکورد متادیتا که عکسشون موجوده: 210122
تعداد نهایی subset: 1100
تعداد کشورهای متفاوت: 219
✅ subset ساخته شد در: C:\Users\ramti\Desktop\ramtin legion pc\projects\myproject\datasets\subset_test


In [ ]:
# The old baseline experiment is now maintained in codes/run_baseline.py.
# It uses the shared project paths and does not require machine-specific paths.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] CLIPVisionModel LOAD REPORT from: laion/CLIP-ViT-L-14-DataComp.XL-s13B-b90K
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0.

Loading weights from local directory


100%|██████████| 600/600 [05:29<00:00,  1.82it/s]

✅ اجرای baseline تموم شد، نتایج ذخیره شد در: C:\Users\ramti\Desktop\ramtin legion pc\projects\myproject\results\clean\baseline_predictions.csv


In [ ]:
# run_baseline.py (نسخه‌ی بهینه‌شده با batch processing)

import sys, os
from pathlib import Path

PROJECT_ROOT = Path.cwd()
CODES_DIR = PROJECT_ROOT / "codes"
OSV5M_REPO = CODES_DIR / "github-osv5m" / "osv5m"
SUBSET_DIR = PROJECT_ROOT / "datasets" / "subset_test"
BASELINE_PATH = CODES_DIR / "baseline-huggingface" / "baseline"
RESULTS_DIR = PROJECT_ROOT / "results"
sys.path.append(str(OSV5M_REPO))
os.chdir(OSV5M_REPO)

import pandas as pd
import torch
from PIL import Image
from tqdm import tqdm
from models.huggingface import Geolocalizer

BATCH_SIZE = 16

geoloc = Geolocalizer.from_pretrained(BASELINE_PATH)
geoloc.eval()

df = pd.read_csv(SUBSET_DIR / "subset_metadata.csv")
df["id_str"] = df["id"].astype(str)

results = []
with torch.no_grad():
    for start in tqdm(range(0, len(df), BATCH_SIZE)):
        batch_rows = df.iloc[start:start+BATCH_SIZE]
        imgs = []
        valid_rows = []
        for _, row in batch_rows.iterrows():
            img_path = SUBSET_DIR / "images" / (row["id_str"] + ".jpg")
            if not img_path.exists():
                continue
            img = Image.open(img_path).convert("RGB")
            imgs.append(geoloc.transform(img))
            valid_rows.append(row)

        if not imgs:
            continue

        x = torch.stack(imgs)
        gps = geoloc(x)

        for i, row in enumerate(valid_rows):
            pred_lat = torch.rad2deg(gps[i][0]).item()
            pred_lon = torch.rad2deg(gps[i][1]).item()
            results.append({
                "id": row["id"],
                "true_lat": row["latitude"],
                "true_lon": row["longitude"],
                "pred_lat": pred_lat,
                "pred_lon": pred_lon,
                "country": row["country"],
            })

out_df = pd.DataFrame(results)
out_path = RESULTS_DIR / "clean" / "baseline_predictions.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
out_df.to_csv(out_path, index=False)
print("✅ اجرای baseline تموم شد، نتایج ذخیره شد در:", out_path)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] CLIPVisionModel LOAD REPORT from: laion/CLIP-ViT-L-14-DataComp.XL-s13B-b90K
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
visual_projection.weight                                     | UNEXPECTED |  | 
text_model.encoder.layers.{0.

Loading weights from local directory


100%|██████████| 69/69 [06:42<00:00,  5.84s/it]

✅ اجرای baseline تموم شد، نتایج ذخیره شد در: C:\Users\ramti\Desktop\ramtin legion pc\projects\myproject\results\clean\baseline_predictions.csv
